# Sprint 3 - Tools and MCP for HelioDesk

This Colab is the shared Sprint 3 notebook for the HelioDesk support assistant. It turns the sprint concepts into one inspectable application path: a support specialist needs to decide whether a customer export can be shared with an external auditor.

You will work through three connected tasks:

1. inspect and run a direct authorization tool;
2. classify tool failures and choose safe recovery patterns; and
3. compare a multi-step tool loop with an MCP policy-search boundary.

Most implementation details are hidden behind helper functions so the notebook can stay focused on evidence, decisions, and checks. You can expand hidden cells if you are curious, but the student work happens in the visible cells.

Choose **File > Save a copy in Drive** before editing in Colab.

**Resource type:** Campus educational notebook: sprint-specific business case from the Campus lessons.


## 1. Install the helper core from GitHub

Run this first in Colab. It installs the shared helper core directly from the GitHub `main` branch so the notebook uses the same code as the course scaffold.


In [ ]:
#@title Install helper core from GitHub { display-mode: "form" }
%pip install -q --force-reinstall --no-cache-dir "ms-ai-ml-helper-core @ git+https://github.com/richhiey/ai-app-dev_Mod-A.git@main"


## 2. Set up the notebook

Run the hidden setup cell once after installation. It imports high-level helpers from the shared course package, so the visible notebook stays focused on evidence, decisions, and checks.

The optional live model cell later in the notebook uses OpenRouter. Store `OPENROUTER_API_KEY` in Colab Secrets when possible. The notebook never prints your key.


In [ ]:
#@title Setup: imports and high-level notebook helpers { display-mode: "form" }
from sprint3_tools_mcp import (
    build_checkpoint_evidence,
    build_heliodesk_tool_registry,
    build_task_answer_signals,
    connect_to_heliodesk_policy_mcp,
    display_direct_authorization_check,
    display_failure_evidence,
    display_mcp_validation,
    display_recovery_review,
    display_registered_tools,
    display_scripted_tool_loop,
    inspect_and_display_contracts,
    pretty,
    review_decision_note,
    review_recovery_plan,
    run_direct_authorization_check,
    run_failure_scenarios,
    run_live_tool_loop,
    run_scripted_tool_loop,
    show_case_brief,
    show_setup_status,
    validate_checkpoint_evidence,
    validate_mcp_policy_response,
    assert_direct_authorization_check,
    assert_failure_scenarios,
)


In [ ]:
OPENROUTER_READY = show_setup_status()


## 3. Read the case brief

The same business question appears throughout the notebook. Keep the two evidence sources separate: current authorization state and handbook policy evidence.


In [ ]:
case_brief = show_case_brief()
pretty(case_brief)


## 4. Inspect the schemas sent to the model

A tool is application code plus a model-facing contract. The model may propose a call, but the application validates the arguments, executes the handler, and decides whether the result is safe to use.

Run the next two cells, then answer Task A from the outputs.


In [ ]:
registry = build_heliodesk_tool_registry()
display_registered_tools(registry)


In [ ]:
openrouter_tool_schemas, contract_report = inspect_and_display_contracts(registry)


## 5. Execute a valid direct tool call

This direct call checks current authorization for the HelioDesk case. Notice what the result proves, and what it does not prove.


In [ ]:
authorization_result, authorization_payload = run_direct_authorization_check(registry)
display_direct_authorization_check(authorization_result, authorization_payload)
assert_direct_authorization_check(authorization_result, authorization_payload)


## 6. Handle malformed and failed tool calls

A failed call is still information. The application should classify what happened before the result reaches a downstream answer.

Run the scenario cell, then edit the recovery choices in the following **Your turn** cell only if you want to test another plan.


In [ ]:
failure_evidence, failure_raw = run_failure_scenarios(registry)
display_failure_evidence(failure_evidence, failure_raw, focus_case="missing_data")
assert_failure_scenarios(failure_evidence)


### Your turn: choose recovery patterns

Keep each recovery decision bounded. Do not treat missing, failed, or permission-blocked tool output as approval to continue.


In [ ]:
failure_decisions = {
    "invalid_input": "ask_for_missing_input",
    "malformed_response": "reject_and_log",
    "missing_data": "reject_and_log",
    "timeout": "retry_once_then_route",
    "empty_result": "honest_no_result",
    "permission_failure": "route_to_authorized_owner",
}

recovery_review = review_recovery_plan(failure_evidence, failure_decisions)
display_recovery_review(recovery_review)
assert all(row["passes"] for row in recovery_review)


## 7. Let the model run a multi-step tool loop

Multi-step tool reasoning means the next call depends on the previous result. This deterministic client stands in for the model so everyone gets the same trace: first current authorization, then policy evidence, then a final support-facing answer.


In [ ]:
scripted_run, tool_trace = run_scripted_tool_loop(registry)
display_scripted_tool_loop(scripted_run, tool_trace)

assert [result.name for result in scripted_run.tool_results] == [
    "check_export_authorization",
    "search_heliodesk_policy",
]


### Optional: run a live OpenRouter tool loop

The scripted trace proves the application boundary without spending model credits. If your course key is available, you can also let a real OpenRouter model see the same schemas and propose calls. Model wording can vary, so judge the trace by tool calls, arguments, validation, and evidence use rather than exact prose.


In [ ]:
#@title Optional live model check { display-mode: "form" }
RUN_LIVE_OPENROUTER = False  #@param {type:"boolean"}

live_run = run_live_tool_loop(registry, enabled=RUN_LIVE_OPENROUTER)


## 8. Connect to the MCP server

Model Context Protocol, or MCP, is a standard connection boundary between an AI host and a server that exposes capabilities. This cell creates the helper-core MCP server object, then launches the installed helper server over `stdio` and calls its `keyword_search` tool with HelioDesk policy snippets.


In [ ]:
mcp_demo, mcp_summary = await connect_to_heliodesk_policy_mcp()
pretty(mcp_summary)


## 9. Validate the MCP response

An MCP connection proves only that the boundary exists. The host application still validates the advertised capability and returned content before using it downstream.


In [ ]:
validated_mcp_rows, mcp_downstream_note = validate_mcp_policy_response(mcp_demo)
display_mcp_validation(validated_mcp_rows, mcp_downstream_note)

assert validated_mcp_rows
assert any("authorization" in row["text"].lower() for row in validated_mcp_rows)


### Your turn: write the decision note

Before collecting final evidence, write a short note answering Task C. Explain one contract decision, one failure-handling decision, and how the MCP result differs from the direct authorization lookup.

The sample note below is a strong answer. Keep it if it matches your reasoning, or edit it in your own words without removing the required ideas.


In [ ]:
decision_note = """
I kept the authorization lookup read-only and required confirmed workspace, requester, and request type fields.
I would not treat an empty result as approval because it only proves that no matching record was found.
A timeout may be retried once because this fixture is read-only, but a write action would need stronger protection.
The MCP keyword_search result gives handbook evidence, while the direct tool gives current authorization state.
The downstream response should use both pieces of evidence and stop when either boundary is missing or unsafe.
""".strip()

decision_note_review = review_decision_note(decision_note)
pretty(decision_note_review)
assert decision_note_review["passes"]


## 10. Sprint 3 checkpoint answers and evidence

Use the next cell as the final guided check. It gathers the notebook evidence, shows the three task answer signals, and asserts that the required notebook outputs are present.

The checkpoint is complete when:

- **Task A:** the direct authorization tool is read-only, requires confirmed fields, rejects unsafe arguments, and returns current authorization evidence `AUTH-8841` for the HelioDesk case.
- **Task B:** every failure family has a safe recovery pattern, and every row in `recovery_review` passes.
- **Task C:** the tool loop uses current authorization first, then policy search; MCP `keyword_search` supplies policy evidence; and the downstream note keeps current-state evidence separate from policy evidence.


In [ ]:
TASK_ANSWER_SIGNALS = build_task_answer_signals()
checkpoint_evidence = build_checkpoint_evidence(
    contract_report=contract_report,
    authorization_result=authorization_result,
    authorization_payload=authorization_payload,
    failure_evidence=failure_evidence,
    recovery_review=recovery_review,
    scripted_run=scripted_run,
    mcp_demo=mcp_demo,
    validated_mcp_rows=validated_mcp_rows,
    mcp_downstream_note=mcp_downstream_note,
    decision_note_review=decision_note_review,
)
TASK_OUTPUT_SUMMARY = checkpoint_evidence

pretty({"answer_signals": TASK_ANSWER_SIGNALS, "your_task_outputs": TASK_OUTPUT_SUMMARY})
validate_checkpoint_evidence(
    authorization_payload=authorization_payload,
    recovery_review=recovery_review,
    scripted_run=scripted_run,
    validated_mcp_rows=validated_mcp_rows,
    decision_note_review=decision_note_review,
)

print("Compare your Task A, Task B, and Task C answers with the answer signals above and the notebook outputs you generated.")


## Optional local inspector

The notebook already connects to the helper MCP server over `stdio`. If you want to inspect the same server locally with the MCP Inspector, run this from a local clone:

```bash
git clone https://github.com/richhiey/ai-app-dev_Mod-A.git
cd ai-app-dev_Mod-A
python -m pip install -e ".[dev]"
mcp dev src/mcp_server.py
```

In the local inspector, call `keyword_search` with the HelioDesk policy snippets from this notebook. Treat the inspector output as UI evidence only after you have captured it from the actual tool.
